In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import spikeinterface.full as si
import spikeinterface.preprocessing as spre
import spikeinterface.widgets as sw
from pathlib import Path

from utils import (
    build_tetrode_probe, validate_probe_mapping,
    list_folders_with_pattern, si_folder_valid, safe_rmtree,
    run_sorter_pipeline,
)

In [ ]:
# ── PATHS  (edit these) ─────────────────────────────────────────────────────
# openephys_folder : root folder that contains 'Record Node ...' subdirectories
try:
    openephys_folder   # allow override from a main_pipeline notebook
    base_folder
except NameError:
    openephys_folder = Path(r'D:\Data\2023-10-16_15-46-26')
    base_folder      = Path(r'D:\Data\2023-10-16_15-46-26\spikeinterface_output')

base_folder.mkdir(parents=True, exist_ok=True)

# ── RECORDING PARAMETERS ────────────────────────────────────────────────────
stream_name      = 'Signals CH'   # run si.get_neo_streams('openephys', openephys_folder) to list
block_index      = 0              # which experiment block to load (0 = first)
freq_min         = 300.           # highpass cutoff (Hz)
cref_operator    = 'median'
cref_reference   = 'global'

# ── SORTER ──────────────────────────────────────────────────────────────────
try:
    sorter = SORTER_KS   # override from main_pipeline
except NameError:
    sorter = 'kilosort4' # 'kilosort2_5' | 'kilosort4' | 'spykingcircus2'

# ── JOB PARALLELISM ─────────────────────────────────────────────────────────
job_kwargs = dict(n_jobs=4, chunk_duration='1s', progress_bar=True)

In [ ]:
# ── PROBE DEFINITION ────────────────────────────────────────────────────────
# 8-tetrode array (32 ch).  build_tetrode_probe() is in utils.py.
# channel_indices: device ADC indices mapped to probe contacts, in contact order.
# Bristol layout.
# Change these numbers if your rig uses a different wiring.

channel_indices_bristol = [
    15, 31, 16,  0,
    13, 29, 18,  2,
    14, 30, 17,  1,
     7, 23, 24,  8,
     6, 22, 25,  9,
     5, 21, 26, 10,
     4, 20, 27, 11,
     3, 19, 28, 12,
]

probe = build_tetrode_probe(
    channel_indices  = channel_indices_bristol,
    n_tetrodes       = 8,
    tetrode_pitch_um = 50.3,
)
print(probe)

In [ ]:
# ── LOAD ────────────────────────────────────────────────────────────────────
raw_rec = si.read_openephys(
    openephys_folder,
    stream_name      = stream_name,
    block_index      = block_index,
    load_sync_channel = False,
)

# Attach probe + set device indices on the recording object
raw_rec = raw_rec.set_probe(probe, group_mode='by_shank')
if raw_rec.get_property('device_channel_indices') is None:
    raw_rec.set_property('device_channel_indices', np.asarray(probe.device_channel_indices))

# Validate mapping immediately after loading
_, _ = validate_probe_mapping(
    raw_rec, probe,
    channel_indices_expected = channel_indices_bristol,
    csv_path = base_folder / 'channel_map_full.csv',
    label    = 'full',
)

fs       = raw_rec.get_sampling_frequency()
num_chan = raw_rec.get_num_channels()
print(raw_rec)

In [ ]:
# ── OPTIONAL: KEEP ONLY SPECIFIC CHANNELS ───────────────────────────────────
# Uncomment and edit the list below if you want to drop channels before
# bad-channel detection (e.g. channels you already know are broken).

# raw_rec = raw_rec.channel_slice([
#     'CH1','CH2','CH3','CH4','CH5','CH7','CH8',
#     'CH11','CH12','CH13','CH15','CH17','CH18',
#     'CH21','CH22','CH23','CH24','CH25','CH26','CH27',
#     'CH31','CH32',
# ])

In [ ]:
# ── ARTIFACT REMOVAL ────────────────────────────────────────────────────────
# PASS 1 — TTL-triggered stimulation artifacts
# Reads digital events from the OpenEphys event file.
# Falls back to timestamps.csv if no TTL events are found.

import spikeinterface.extractors as se

list_triggers = []
for seg in range(raw_rec.get_num_segments()):
    events = se.read_openephys_event(
        folder_path = openephys_folder,
        block_index = block_index,
    )
    triggers = events.get_event_times(
        channel_id    = 'Rhythm FPGA TTL Input',
        segment_index = seg,
    )
    # shift triggers to be relative to segment start (important for multi-segment)
    seg_t0      = raw_rec.get_times(segment_index=seg)[0]
    new_triggers = (triggers - seg_t0) * fs
    list_triggers.append(new_triggers.astype(int))
    print(f'  Segment {seg}: {len(new_triggers)} TTL triggers found')

# Fallback: if no TTL events found, try timestamps.csv
if len(list_triggers) == 1 and len(list_triggers[0]) == 0:
    node_folders = list_folders_with_pattern(str(openephys_folder), r'^Record Node \d+$')
    node_folders.sort()
    block_folders = list_folders_with_pattern(
        str(openephys_folder / node_folders[0]), r'^experiment\d+$')
    ts_csv = (
        openephys_folder / node_folders[0] / block_folders[block_index]
        / 'timestamps.csv'
    )
    if ts_csv.exists():
        import pandas as pd
        df_ts = pd.read_csv(ts_csv, header=None)
        frames = (df_ts[0] * fs).astype(int).tolist()
        list_triggers = [frames]
        print(f'  Fallback: loaded {len(frames)} triggers from {ts_csv.name}')
    else:
        print('  No TTL events and no timestamps.csv found - skipping pass-1 artifact removal.')

rec_no_ttl = spre.remove_artifacts(
    raw_rec,
    list_triggers = list_triggers,
    ms_before = 2,
    ms_after  = 5,
    mode      = 'zeros',
)

# PASS 2 — manually identified noise transients (interpolated, less harsh)
# Add the absolute timestamps (in seconds) of any remaining bad transients.
# Leave the list empty to skip this step.

artifact_times_s = [
    # 556.61, 728.67, 934.73,   # example values - edit or clear this list
]

if artifact_times_s:
    ms_before_2, ms_after_2 = 20, 40
    pre_fr  = int(np.rint(ms_before_2 * 1e-3 * fs))
    post_fr = int(np.rint(ms_after_2  * 1e-3 * fs))

    list_triggers_2 = []
    for seg in range(rec_no_ttl.get_num_segments()):
        seg_t0  = rec_no_ttl.get_times(segment_index=seg)[0]
        seg_len = rec_no_ttl.get_num_frames(segment_index=seg)
        frames  = np.rint((np.array(artifact_times_s) - seg_t0) * fs).astype(np.int64)
        frames  = np.unique(np.sort(frames))
        in_bounds = (frames - pre_fr >= 0) & (frames + post_fr < seg_len)
        list_triggers_2.append(frames[in_bounds])

    rec_no_artifacts = spre.remove_artifacts(
        rec_no_ttl,
        list_triggers = list_triggers_2,
        ms_before = ms_before_2,
        ms_after  = ms_after_2,
        mode      = 'linear',
    )
    print('Pass-2 artifact removal done.')
else:
    rec_no_artifacts = rec_no_ttl
    print('No manual artifact times specified - skipping pass-2.')

In [ ]:
# ── BAD CHANNELS + PREPROCESSING ────────────────────────────────────────────
# Optional: load a pre-saved list of bad channels from bad_channels.json
# (same format as the original pipeline)

bad_channel_ids = []
bad_ch_json = openephys_folder / 'bad_channels.json'
if bad_ch_json.exists():
    with open(bad_ch_json) as f:
        bad_channel_ids = json.load(f).get('bad_channels', [])
    bad_channel_ids = sorted(set(map(int, bad_channel_ids)))
    print('Bad channels loaded from file:', bad_channel_ids)

rec_hp = si.highpass_filter(rec_no_artifacts, freq_min=freq_min)
detected_bad, _ = si.detect_bad_channels(rec_hp)
bad_channel_ids = sorted(set(bad_channel_ids) | set(detected_bad.tolist()))
print('Bad channels (file + auto-detected):', bad_channel_ids)

rec_clean = rec_hp.remove_channels(bad_channel_ids)

# Validate mapping after bad-channel removal
_, _ = validate_probe_mapping(
    rec_clean, probe,
    csv_path = base_folder / 'channel_map_after_bad_removal.csv',
    label    = 'after-bad',
)

rec = si.common_reference(rec_clean, operator=cref_operator, reference=cref_reference)
print(rec)

In [ ]:
# ── TRACE VISUALISATION ─────────────────────────────────────────────────────
import re
_nat_key = lambda s: [int(t) if t.isdigit() else t.lower()
                      for t in re.split(r'(\d+)', s)]
_sorted_ids = sorted(raw_rec.channel_ids, key=_nat_key)
_good_ids   = [c for c in _sorted_ids if c in set(rec.channel_ids)]

raw_2s = raw_rec.frame_slice(0, int(2 * fs))
rec_2s = rec.frame_slice(0, int(2 * fs))

sw.plot_traces(raw_2s, channel_ids=_sorted_ids[:10], mode='line', backend='matplotlib')
plt.suptitle('Raw traces (first 2 s, first 10 ch)')
plt.tight_layout(); plt.show()

sw.plot_traces({'raw': raw_2s, 'preprocessed': rec_2s},
               channel_ids=_good_ids[:6], mode='line', backend='matplotlib')
plt.suptitle('Raw vs preprocessed')
plt.tight_layout(); plt.show()

In [ ]:
#  SAVE PREPROCESSED BINARY 
# Saving to binary before sorting avoids repeatedly re-running the
# preprocessing chain.  Skipped automatically if a valid cache exists.

_preprocess_dir = base_folder / 'preprocess'

if si_folder_valid(_preprocess_dir):
    rec = si.load(_preprocess_dir)
    print(f'Loaded cached preprocessed recording from {_preprocess_dir.name!r}')
else:
    if _preprocess_dir.exists():
        safe_rmtree(_preprocess_dir)
    rec = rec.save(folder=_preprocess_dir, format='binary', **job_kwargs)
    print(f'Preprocessed recording saved to {_preprocess_dir.name!r}')

In [ ]:
# SPIKE SORTING + ANALYSIS + PHY EXPORT ───────────────────────────────────
# run_sorter_pipeline() is defined in utils.py.
# It runs the sorter, builds the SortingAnalyzer, exports to Phy,
# and writes quality_metrics.csv / template_metrics.csv / metrics_merged.csv.

analyzer = run_sorter_pipeline(
    base_folder  = base_folder,
    sorter_name  = sorter,
    recording    = rec,
    analyzer_format     = 'binary',   # change to 'memory' to skip saving
    compute_pc_features = True,       # set False for very high channel-count probes
    metric_names = ['snr', 'isi_violation', 'presence_ratio'],
    job_kwargs   = job_kwargs,
)

In [ ]:
# ── QUICK RESULTS OVERVIEW ──────────────────────────────────────────────────
sorting   = analyzer.sorting
unit_ids  = sorting.get_unit_ids()
duration  = sorting.get_total_duration()

print(f'Units found: {len(unit_ids)}')
for uid in unit_ids:
    spikes = sorting.get_unit_spike_train(uid, segment_index=0)
    print(f'  Unit {uid:>4}: {len(spikes):>5} spikes  |  {len(spikes)/duration:.1f} Hz')

si.plot_autocorrelograms(sorting)
plt.suptitle('Autocorrelograms'); plt.tight_layout(); plt.show()

si.plot_unit_templates(analyzer)
plt.suptitle('Unit templates'); plt.tight_layout(); plt.show()

si.plot_amplitudes(analyzer)
plt.suptitle('Amplitudes over time'); plt.tight_layout(); plt.show()

In [ ]:
# ── INTERACTIVE VIEWER ──────────────────────────────────────────────────────
# spikeinterface_gui shows waveforms, ISI, amplitudes, probe map.
# Close this window before re-running any cell that saves to disk on Windows.

import spikeinterface_gui
spikeinterface_gui.run_mainwindow(analyzer)